In [1]:
import pandas as pd
df=pd.read_csv("bridges.csv")
df.shape, df.head()

((108, 13),
    Id river  location  erected   purpose  length  lanes clear-g   t-or-d  \
 0  E1     M       3.0     1818   HIGHWAY     NaN    2.0       N  THROUGH   
 1  E2     A      25.0     1819   HIGHWAY  1037.0    2.0       N  THROUGH   
 2  E3     A      39.0     1829  AQUEDUCT     NaN    1.0       N  THROUGH   
 3  E5     A      29.0     1837   HIGHWAY  1000.0    2.0       N  THROUGH   
 4  E6     M      23.0     1838   HIGHWAY     NaN    2.0       N  THROUGH   
 
   material   span rel-l  type  
 0     WOOD  SHORT     S  WOOD  
 1     WOOD  SHORT     S  WOOD  
 2     WOOD    NaN     S  WOOD  
 3     WOOD  SHORT     S  WOOD  
 4     WOOD    NaN     S  WOOD  )

In [2]:
df["length"].describe()

count      81.000000
mean     1567.469136
std       747.491523
min       804.000000
25%      1000.000000
50%      1300.000000
75%      2000.000000
max      4558.000000
Name: length, dtype: float64



### Plausible Assumption
I am going to assume that the plausible bridge length is any length that falls between 1.5 IQR from the first and third quartiles.



In [3]:
## Starting with verifying rows with NA value for length
print(f"{df['length'].isna().sum()} blank rows")
df[df["length"].isna()]


27 blank rows


,Id,river,location,erected,purpose,length,lanes,clear-g,t-or-d,material,span,rel-l,type
0,E1,M,3.0,1818,HIGHWAY,NaN,2.0,N,THROUGH,WOOD,SHORT,S,WOOD
2,E3,A,39.0,1829,AQUEDUCT,NaN,1.0,N,THROUGH,WOOD,NaN,S,WOOD
4,E6,M,23.0,1838,HIGHWAY,NaN,2.0,N,THROUGH,WOOD,NaN,S,WOOD
8,E10,A,39.0,1848,AQUEDUCT,NaN,1.0,N,DECK,WOOD,NaN,S,WOOD
10,E12,A,39.0,1853,RR,NaN,2.0,N,DECK,WOOD,NaN,S,WOOD
12,E13,A,33.0,1856,HIGHWAY,NaN,2.0,N,THROUGH,WOOD,NaN,S,WOOD
13,E15,A,28.0,1857,RR,NaN,2.0,N,THROUGH,WOOD,NaN,S,WOOD
19,E21,M,16.0,1874,RR,NaN,2.0,NaN,THROUGH,IRON,NaN,NaN,SIMPLE-T
22,E24,O,45.0,1878,RR,NaN,2.0,G,NaN,STEEL,NaN,NaN,SIMPLE-T
23,E25,M,10.0,1882,RR,NaN,2.0,G,NaN,STEEL,NaN,NaN,SIMPLE-T


In [4]:
## Dropping rows with NA in length and verifying the drop
df = df.dropna(subset=['length'])
df[df["length"].isna()]

,Id,river,location,erected,purpose,length,lanes,clear-g,t-or-d,material,span,rel-l,type


In [5]:
## Calculating the Q1 and Q3 percentiles
Q1 = df["length"].quantile(0.25)
Q3 = df["length"].quantile(0.75)

## Calculating IQR
IQR = Q3-Q1

## Defining Plausible boundaries
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

## Filtering Implausible data
implausible_filter = (df['length'] < lower) | (df['length'] > upper)
flagged_data = df[implausible_filter]

print(f"{len(flagged_data)} implausible rows")
flagged_data

3 implausible rows


,Id,river,location,erected,purpose,length,lanes,clear-g,t-or-d,material,span,rel-l,type
31,E34,O,41.0,1888,RR,4558.0,2.0,G,THROUGH,STEEL,LONG,F,SIMPLE-T
44,E46,A,37.0,1897,RR,4000.0,2.0,G,DECK,STEEL,LONG,F,SIMPLE-T
104,E91,O,44.0,1975,HIGHWAY,3756.0,6.0,G,THROUGH,STEEL,LONG,F,ARCH


## Handling implausible Rows
I am going to leave those rows alone, because even though they are outside my assumed plausible range for the bridges, based on the IQR of Q1 and Q3, it is not actually an impossible length for a bridge. A google search of bridges in Pittsburg shows that the longest bridge is actually much longer than the ones we have recorded in our dataset and recognizing that the material used is steel, which would be strong enough to build a bridge of that length, leaving these rows in is the best choice, because there is a very small chance those numbers are inaccurate. 

## Cleaning Log
- Removed all rows that had no value in the length column

#### GitHub URL
